In [1]:
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
from numpy import array, nan, inf, float32  # noqa: F401

In [2]:
from dataclasses import dataclass


@dataclass
class RangeBounds:
    min: float
    max: float
    min_abs: float


@dataclass
class BitInformationBounds:
    percentile: list[float]
    keepbits: list[int]
    linquant_absolute: list[float]
    bitround_relative: list[float]


@dataclass
class EnsembleSpreadBounds:
    percentile: list[float]
    absolute: list[float]
    relative: list[float]
    quadratic: list[float]


@dataclass
class MinimumBounds:
    percentile: list[float]
    mean_bin_edges: list[float]
    mean_bin_counts: list[int]
    mean_bin_spread_bounds: list[list[float]]


@dataclass
class CompressionRatios:
    percentile: list[float]
    absolute: list[float]
    relative: list[float]
    quadratic: list[float]

In [3]:
def parse_single(path: Path):
    with Path(path).open() as f:
        for line in f.readlines():
            var, rest = line.split(": ", 1)
            info = eval(rest)

            yield var, info

In [4]:
def parse_pressure(path: Path):
    with Path(path).open() as f:
        ls = iter(f.readlines())

        line = None

        while True:
            if line is None:
                try:
                    line = next(ls)
                except Exception:
                    break

            var, rest = line.split(":", 1)

            while True:
                try:
                    line = next(ls)
                except Exception:
                    line = None
                    break

                if len(line.strip()) == 0:
                    continue

                if not line.startswith(" - "):
                    break

                level, rest = line[len(" - ") :].split(": ", 1)
                info = eval(rest)

                yield var, level, info

In [5]:
def parse_float(x: str):
    if len(x.strip()) == 0:
        return np.nan
    return float(x.replace(",", "."))

In [6]:
expert = pd.read_csv("expert.csv")

In [7]:
table = SimpleNamespace(
    var=[],
    level=[],
    percentile=[],
    min=[],
    range=[],
    max=[],
    lpbits=[],
    lpabsolute=[],
    lprelative=[],
    brlqabsolute=[],
    brabsolute=[],
    brrelative=[],
    esabsolute=[],
    esrelative=[],
    esquadratic=[],
    cabsolute=[],
    crelative=[],
    cquadratic=[],
    pick=[],
    crabsolute=[],
    crrelative=[],
    crquadratic=[],
    exabsl1=[],
    exabslinfty=[],
    exrell1=[],
    exrellinfty=[],
)

for (
    (rvar, rinfo),
    (lvar, linfo),
    (bvar, binfo),
    (evar, einfo),
    (mvar, minfo),
    (cvar, cinfo),
) in zip(
    parse_single("range-single.txt"),
    parse_single("linear-packing-single.txt"),
    parse_single("bit-information-single.txt"),
    parse_single("ensemble-spread-single.txt"),
    parse_single("minimum-single.txt"),
    parse_single("compression-ratio-single.txt"),
):
    assert all(v == rvar for v in [rvar, lvar, bvar, evar, mvar, cvar])
    assert (
        binfo.percentile == einfo.percentile
        and minfo.percentile == einfo.percentile
        and cinfo.percentile == einfo.percentile
    )

    for p, k, a, r, q, b, ca, cr, cq in zip(
        binfo.percentile,
        binfo.keepbits,
        einfo.absolute,
        einfo.relative,
        einfo.quadratic,
        minfo.mean_bin_spread_bounds,
        cinfo.absolute,
        cinfo.relative,
        cinfo.quadratic,
    ):
        table.var.append(bvar)
        table.level.append("single")
        table.percentile.append(
            np.format_float_positional(100 * p, unique=True, trim="-") + "%"
        )

        table.min.append(f"{rinfo.min:.2e}")
        table.range.append(f"{rinfo.max - rinfo.min:.2e}")
        table.max.append(f"{rinfo.max:.2e}")

        table.lpbits.append(f"{linfo}")
        table.lpabsolute.append(f"{(rinfo.max - rinfo.min) / np.exp2(linfo) * 0.5:.1e}")
        table.lprelative.append(
            f"{((rinfo.max - rinfo.min) / np.exp2(linfo) * 0.5) / rinfo.max:.2%}"
        )

        table.brlqabsolute.append(
            f"{(rinfo.max - rinfo.min) / np.exp2(k - 9) * 0.5:.1e}"
        )
        table.brabsolute.append(
            f"{np.exp2(int(np.log2(rinfo.min_abs))) / np.exp2(k - 9):.1e}"
        )
        table.brrelative.append(f"{np.exp2(-(k - 9)) * 0.5:.2%}")

        table.esabsolute.append(f"{a:.1e}")
        table.esrelative.append(f"{r:.2%}")
        table.esquadratic.append(f"{q:.1e}")

        cabs = f"{np.nansum(np.abs(np.array(b) - a) * minfo.mean_bin_counts) / np.sum(minfo.mean_bin_counts):.1e}"
        crel = f"{np.nansum(np.abs(np.array(b) - np.abs((np.array(minfo.mean_bin_edges[:-1]) + np.array(minfo.mean_bin_edges[1:])) / 2) * r) * minfo.mean_bin_counts) / np.sum(minfo.mean_bin_counts):.1e}"
        cqua = f"{np.nansum(np.abs(np.array(b) - (q - q * (-1.0 + 2.0 * (np.array((np.array(minfo.mean_bin_edges[:-1]) + np.array(minfo.mean_bin_edges[1:])) / 2) - rinfo.min) / (rinfo.max - rinfo.min)) ** 2)) * minfo.mean_bin_counts) / np.sum(minfo.mean_bin_counts):.1e}"

        table.cabsolute.append(cabs)
        table.crelative.append(crel)
        table.cquadratic.append(cqua)

        bounds = [("absolute", cabs), ("relative", crel), ("quadratic", cqua)]
        table.pick.append(sorted(bounds, key=lambda x: float(x[1]))[0][0])

        table.crabsolute.append(f"{ca:.2e}")
        table.crrelative.append(f"{cr:.2e}")
        table.crquadratic.append(f"{cq:.2e}")

        ex = expert[
            (expert["Short Name"] == bvar) & (expert["Level"] == "Single/Surface")
        ]
        if len(ex) > 0:
            assert len(ex) == 1
            ex = ex.iloc[0]
            table.exabsl1.append(f"{parse_float(str(ex['Absolute L1 error']))}")
            table.exabslinfty.append(f"{parse_float(str(ex['Absolute Linfty error']))}")
            table.exrell1.append(f"{parse_float(str(ex['Relative L1 error']))}")
            table.exrellinfty.append(f"{parse_float(str(ex['Relative Linfty error']))}")
        else:
            table.exabsl1.append(np.nan)
            table.exabslinfty.append(np.nan)
            table.exrell1.append(np.nan)
            table.exrellinfty.append(np.nan)

for (
    (rvar, rlevel, rinfo),
    (lvar, llevel, linfo),
    (bvar, blevel, binfo),
    (evar, elevel, einfo),
    (mvar, mlevel, minfo),
    (cvar, clevel, cinfo),
) in zip(
    parse_pressure("range-pressure.txt"),
    parse_pressure("linear-packing-pressure.txt"),
    parse_pressure("bit-information-pressure.txt"),
    parse_pressure("ensemble-spread-pressure.txt"),
    parse_pressure("minimum-pressure.txt"),
    parse_pressure("compression-ratio-pressure.txt"),
):
    assert all(v == rvar for v in [rvar, lvar, bvar, evar, mvar, cvar])
    assert all(
        level == rlevel for level in [rlevel, llevel, blevel, elevel, mlevel, clevel]
    )
    assert (
        binfo.percentile == einfo.percentile
        and minfo.percentile == einfo.percentile
        and cinfo.percentile == einfo.percentile
    )

    for p, k, a, r, q, b, ca, cr, cq in zip(
        binfo.percentile,
        binfo.keepbits,
        einfo.absolute,
        einfo.relative,
        einfo.quadratic,
        minfo.mean_bin_spread_bounds,
        cinfo.absolute,
        cinfo.relative,
        cinfo.quadratic,
    ):
        table.var.append(bvar)
        table.level.append(blevel)
        table.percentile.append(
            np.format_float_positional(100 * p, unique=True, trim="-") + "%"
        )

        table.min.append(f"{rinfo.min:.2e}")
        table.range.append(f"{rinfo.max - rinfo.min:.2e}")
        table.max.append(f"{rinfo.max:.2e}")

        table.lpbits.append(f"{linfo}")
        table.lpabsolute.append(f"{(rinfo.max - rinfo.min) / np.exp2(linfo) * 0.5:.1e}")
        table.lprelative.append(
            f"{((rinfo.max - rinfo.min) / np.exp2(linfo) * 0.5) / rinfo.max:.2%}"
        )

        table.brlqabsolute.append(
            f"{(rinfo.max - rinfo.min) / np.exp2(k - 9) * 0.5:.1e}"
        )
        table.brabsolute.append(
            f"{np.exp2(np.floor(np.log2(rinfo.min_abs))) / np.exp2(k - 9):.1e}"
        )
        table.brrelative.append(f"{np.exp2(-(k - 9)) * 0.5:.2%}")

        table.esabsolute.append(f"{a:.1e}")
        table.esrelative.append(f"{r:.2%}")
        table.esquadratic.append(f"{q:.1e}")

        cabs = f"{np.nansum(np.abs(np.array(b) - a) * minfo.mean_bin_counts) / np.sum(minfo.mean_bin_counts):.1e}"
        crel = f"{np.nansum(np.abs(np.array(b) - np.abs((np.array(minfo.mean_bin_edges[:-1]) + np.array(minfo.mean_bin_edges[1:])) / 2) * r) * minfo.mean_bin_counts) / np.sum(minfo.mean_bin_counts):.1e}"
        cqua = f"{np.nansum(np.abs(np.array(b) - (q - q * (-1.0 + 2.0 * (np.array((np.array(minfo.mean_bin_edges[:-1]) + np.array(minfo.mean_bin_edges[1:])) / 2) - rinfo.min) / (rinfo.max - rinfo.min)) ** 2)) * minfo.mean_bin_counts) / np.sum(minfo.mean_bin_counts):.1e}"

        table.cabsolute.append(cabs)
        table.crelative.append(crel)
        table.cquadratic.append(cqua)

        bounds = [("absolute", cabs), ("relative", crel), ("quadratic", cqua)]
        table.pick.append(sorted(bounds, key=lambda x: float(x[1]))[0][0])

        table.crabsolute.append(f"{ca:.2e}")
        table.crrelative.append(f"{cr:.2e}")
        table.crquadratic.append(f"{cq:.2e}")

        ex = expert[
            (expert["Short Name"] == bvar) & (expert["Level"] == "Pressure+Model")
        ]
        if len(ex) > 0:
            assert len(ex) == 1
            ex = ex.iloc[0]
            table.exabsl1.append(f"{parse_float(str(ex['Absolute L1 error']))}")
            table.exabslinfty.append(f"{parse_float(str(ex['Absolute Linfty error']))}")
            table.exrell1.append(f"{parse_float(str(ex['Relative L1 error']))}")
            table.exrellinfty.append(f"{parse_float(str(ex['Relative Linfty error']))}")
        else:
            table.exabsl1.append(np.nan)
            table.exabslinfty.append(np.nan)
            table.exrell1.append(np.nan)
            table.exrellinfty.append(np.nan)

df_raw = pd.DataFrame.from_dict(table.__dict__).set_index(
    ["var", "level", "percentile"]
)

/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_94409/383756892.py:171: RuntimeWarning: invalid value encountered in scalar divide
  f"{((rinfo.max - rinfo.min) / np.exp2(linfo) * 0.5) / rinfo.max:.2%}"
/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_94409/383756892.py:178: RuntimeWarning: divide by zero encountered in log2
  f"{np.exp2(np.floor(np.log2(rinfo.min_abs))) / np.exp2(k - 9):.1e}"
/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_94409/383756892.py:188: RuntimeWarning: invalid value encountered in divide
  cqua = f"{np.nansum(np.abs(np.array(b) - (q - q * (-1.0 + 2.0 * (np.array((np.array(minfo.mean_bin_edges[:-1]) + np.array(minfo.mean_bin_edges[1:])) / 2) - rinfo.min) / (rinfo.max - rinfo.min)) ** 2)) * minfo.mean_bin_counts) / np.sum(minfo.mean_bin_counts):.1e}"


In [8]:
df_raw.to_csv("table-raw.csv")

In [9]:
df_display = df_raw.rename(
    columns=dict(
        min="minimum",
        range="max - min",
        max="maximum",
        lpbits="#bits (lp)",
        lpabsolute="absolute (lp)",
        lprelative="relative (lp)",
        brlqabsolute="absolute (br+lq)",
        brabsolute="absolute (br)",
        brrelative="relative (br)",
        esabsolute="absolute (es)",
        esrelative="relative (es)",
        esquadratic="quadratic (es)",
        cabsolute="eb_abs cost",
        crelative="eb_rel cost",
        cquadratic="eb_qua cost",
        pick="error bound",
        crabsolute="eb_abs compression",
        crrelative="eb_rel compression",
        crquadratic="eb_qua compression",
        exabsl1="expert L1/mean abs",
        exabslinfty="expert Linf/max abs",
        exrell1="expert L1/mean rel",
        exrellinfty="expert Linf/max rel",
    )
)

In [10]:
df_display.to_csv("table.csv")

In [11]:
from IPython.display import HTML

HTML(df_display.to_html())